# SOS Audio Detection — אימון בקולאב

## לפני הכל:
1. שנו Runtime → **Change runtime type → T4 GPU**
2. הריצו את כל התאים לפי הסדר
3. כל הדאטה נשמר ב-Google Drive שלכן — לא יאבד בין sessions

In [ ]:
# ===== שלב 1: חיבור ל-Google Drive =====
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/SOS-Audio-Detection'
os.makedirs(f'{BASE}/data/raw/scream', exist_ok=True)
os.makedirs(f'{BASE}/data/raw/crying', exist_ok=True)
os.makedirs(f'{BASE}/data/raw/explosion', exist_ok=True)
os.makedirs(f'{BASE}/data/raw/background', exist_ok=True)
os.makedirs(f'{BASE}/data/processed', exist_ok=True)
os.makedirs(f'{BASE}/src', exist_ok=True)
print('תיקיות נוצרו ב-Drive')

In [ ]:
# ===== שלב 2: התקנת ספריות =====
!pip install librosa scikit-learn matplotlib tensorflow -q
print('ספריות מותקנות')

In [ ]:
# ===== שלב 3: הורדת FSD50K ישירות ל-Drive =====
# FSD50K מתארח ב-Zenodo — הורדה ישירה, ללא תלות ביוטיוב
# גודל כולל: ~25GB. אם רוצות רק dev set (קטן יותר, ~10GB): השתמשו ב-URL השני

import os

FSD50K_DIR = f'{BASE}/data/fsd50k'
os.makedirs(FSD50K_DIR, exist_ok=True)

# Dev set בלבד — ~10GB, 4,970 קליפים, תוויות מאומתות על ידי אנשים
# זה מספיק לאימון טוב, אין צורך בכל ה-51k קליפים
if not os.path.exists(f'{FSD50K_DIR}/FSD50K.dev_audio.zip'):
    print('מוריד FSD50K dev set... (יכול לקחת 10-20 דקות)')
    !wget -q --show-progress -O '{FSD50K_DIR}/FSD50K.dev_audio.zip' \
        'https://zenodo.org/record/4060432/files/FSD50K.dev_audio.zip'
    !wget -q -O '{FSD50K_DIR}/FSD50K.ground_truth.zip' \
        'https://zenodo.org/record/4060432/files/FSD50K.ground_truth.zip'
    print('הורדה הושלמה')
else:
    print('FSD50K כבר קיים ב-Drive')

# פריקה
if not os.path.exists(f'{FSD50K_DIR}/FSD50K.dev_audio'):
    print('פורק...')
    !unzip -q '{FSD50K_DIR}/FSD50K.dev_audio.zip' -d '{FSD50K_DIR}/'
    !unzip -q '{FSD50K_DIR}/FSD50K.ground_truth.zip' -d '{FSD50K_DIR}/'
    print('פריקה הושלמה')

In [ ]:
# ===== שלב 4: מיון FSD50K לפי קטגוריות =====
import pandas as pd
import shutil

# מיפוי תוויות FSD50K לקטגוריות שלנו
# התוויות ב-FSD50K משתמשות ב-AudioSet ontology labels
FSD50K_MAPPING = {
    # scream
    'Screaming': 'scream',
    'Shout': 'scream',
    'Yell': 'scream',
    'Crying, sobbing': 'crying',
    'Whimper': 'crying',
    'Sob': 'crying',
    # explosion
    'Explosion': 'explosion',
    'Gunshot, gunfire': 'explosion',
    'Fireworks': 'explosion',
    'Burst, pop': 'explosion',
    # background
    'Traffic noise, roadway noise': 'background',
    'Wind noise (microphone)': 'background',
    'Rain': 'background',
    'Crowd': 'background',
    'Applause': 'background',
    'Music': 'background',
    'Speech': 'background',
    'Bark': 'background',
    'Engine': 'background',
    'Typing': 'background',
}

dev_csv = f'{FSD50K_DIR}/FSD50K.ground_truth/dev.csv'
df = pd.read_csv(dev_csv)
print(f'סה"כ קליפים ב-dev: {len(df)}')
print(df.columns.tolist())
print(df.head(3))

In [ ]:
# המשך מיון — מעתיק קבצים לתיקיות raw
copied = {cat: 0 for cat in ['scream', 'crying', 'explosion', 'background']}
skipped = 0

for _, row in df.iterrows():
    fname = str(row['fname'])
    labels = str(row['labels']).split(',')
    
    category = None
    for lbl in labels:
        lbl = lbl.strip()
        if lbl in FSD50K_MAPPING:
            category = FSD50K_MAPPING[lbl]
            break  # קטגוריית המצוקה גוברת
    
    if category is None:
        skipped += 1
        continue
    
    src = f'{FSD50K_DIR}/FSD50K.dev_audio/{fname}.wav'
    dst = f'{BASE}/data/raw/{category}/fsd50k_{fname}.wav'
    
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        copied[category] += 1

print('הועתקו:', copied)
print(f'דולגו (תוויות לא ממופות): {skipped}')
print('\nסה"כ:')
for cat in ['scream', 'crying', 'explosion', 'background']:
    count = len(os.listdir(f'{BASE}/data/raw/{cat}'))
    print(f'  {cat}: {count}')

In [ ]:
# ===== שלב 5: עיבוד אודיו ל-mel spectrograms =====
import numpy as np
import librosa

RAW_DIR = f'{BASE}/data/raw'
PROCESSED_DIR = f'{BASE}/data/processed'
CATEGORIES = ['scream', 'crying', 'explosion', 'background']
SR = 22050
DURATION = 5.0  # חייב להיות זהה לחלון ב-realtime_detector.py

def process_file(file_path):
    try:
        audio, sr = librosa.load(file_path, sr=SR, duration=DURATION)
        target_length = int(SR * DURATION)
        if len(audio) < target_length:
            audio = np.pad(audio, (0, target_length - len(audio)))
        else:
            audio = audio[:target_length]
        mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128)
        return librosa.power_to_db(mel, ref=np.max)
    except Exception as e:
        print(f'שגיאה ב-{file_path}: {e}')
        return None

total = 0
errors = 0
for category in CATEGORIES:
    category_dir = os.path.join(RAW_DIR, category)
    files = [f for f in os.listdir(category_dir) if f.endswith(('.wav', '.ogg', '.mp3'))]
    for filename in files:
        file_path = os.path.join(category_dir, filename)
        save_path = os.path.join(PROCESSED_DIR, f"{category}_{filename.rsplit('.', 1)[0]}.npy")
        if os.path.exists(save_path):  # דילוג על קבצים שכבר עובדו
            total += 1
            continue
        mel_db = process_file(file_path)
        if mel_db is not None:
            np.save(save_path, mel_db)
            total += 1
        else:
            errors += 1
    print(f'{category}: עובד')

print(f'\nסה"כ עובדו: {total} | שגיאות: {errors}')

In [ ]:
# ===== שלב 6: אימון המודל =====
import json
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow import keras
import matplotlib.pyplot as plt

PROCESSED_DIR = f'{BASE}/data/processed'
CATEGORIES = ['scream', 'crying', 'explosion', 'background']

# טעינת נתונים
X, y = [], []
for label, category in enumerate(CATEGORIES):
    files = [f for f in os.listdir(PROCESSED_DIR) if f.startswith(category + '_')]
    for fname in files:
        mel = np.load(os.path.join(PROCESSED_DIR, fname))
        X.append(mel)
        y.append(label)

X = np.array(X)
print(f'סה"כ דוגמאות: {len(X)} | צורה: {X.shape}')
for i, cat in enumerate(CATEGORIES):
    print(f'  {cat}: {sum(1 for lbl in y if lbl == i)}')

In [ ]:
# נרמול + שמירת ערכים
MEAN = X.mean()
STD = X.std()
X = (X - MEAN) / (STD + 1e-8)
X = X[..., np.newaxis]

norm_stats = {'mean': float(MEAN), 'std': float(STD)}
with open(f'{BASE}/src/norm_stats.json', 'w') as f:
    json.dump(norm_stats, f)
print(f'נרמול נשמר: mean={MEAN:.4f}, std={STD:.4f}')

y_cat = keras.utils.to_categorical(y, num_classes=4)
X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.2, random_state=42)

In [ ]:
# Augmentation
def augment(X_raw, y_raw):
    aug_X, aug_y = [X_raw], [y_raw]
    aug_X.append(X_raw + np.random.normal(0, 0.01, X_raw.shape))  # רעש
    aug_y.append(y_raw)
    aug_X.append(np.roll(X_raw, shift=10, axis=2))  # הזזת זמן
    aug_y.append(y_raw)
    aug_X.append(np.flip(X_raw, axis=2))  # היפוך
    aug_y.append(y_raw)
    aug_X.append(X_raw * np.random.uniform(0.8, 1.2))  # gain
    aug_y.append(y_raw)
    return np.concatenate(aug_X), np.concatenate(aug_y)

X_train, y_train = augment(X_train, y_train)
print(f'אחרי augmentation: {len(X_train)} דוגמאות')

In [ ]:
# בניית מודל + אימון
model = keras.Sequential([
    keras.layers.Input(shape=X.shape[1:]),
    keras.layers.Conv2D(32, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D(2, 2),
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D(2, 2),
    keras.layers.Conv2D(128, (3, 3), activation='relu'),
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(4, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

early_stop = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

In [ ]:
# ===== שמירה + הערכה =====
model.save(f'{BASE}/src/sos_model.keras')
print('המודל נשמר!')

loss, acc = model.evaluate(X_test, y_test)
print(f'דיוק: {acc:.2%}')

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test.argmax(axis=1), y_pred.argmax(axis=1))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CATEGORIES)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig(f'{BASE}/src/training/confusion_matrix.png')
plt.show()
print('Confusion Matrix נשמרה')

## אחרי האימון:
הורידי מה-Drive:
- `SOS-Audio-Detection/src/sos_model.keras`
- `SOS-Audio-Detection/src/norm_stats.json`

העתיקי אותם ל-`src/` בריפו המקומי שלך.